<a href="https://colab.research.google.com/github/madro0131-wq/analysis-everpeak/blob/main/everpeak_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importar librerías y cargar el dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Cargar el DataFrame a procesar: Everpeak
df = pd.read_csv("/datasets/everpeak_retail.csv")

In [ ]:
# Bucle para contar nulos, una por cada columna,
columnas = ["product_category", "quantity", "city"]
for col in columnas:
print(col, "nulos:", df[col].isna().sum())

# 1. - Sentinels: reemplazar marcadores inválidos por NaN

**Llamado desde clean_data**

In [ ]:
# Crear función para estandarizar sentinels

def reemplazar_sentinels_global(df, numeric_cols, text_cols):
    numeric_sentinels = [-999, 999]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].replace(numeric_sentinels, pd.NA)

    text_sentinels = ["?"]
    for col in text_cols:
        df[col] = df[col].replace(text_sentinels, "unknown")
    return df

# 2. - Flags: crear banderas antes de imputar
**Llamado desde clean_data**

In [ ]:
# crear función para crear columnas flags
def crear_flags(df, flags_cols):
    for col in flags_cols:
        nombre_flag = col + "_missing_flag"
        df[nombre_flag] = df[col].isna().astype(int)
    return df

# 3. - Imputar según diagnóstico

- Recibe como entrada un DataFrame y las listas de columnas numéricas, categóricas y de fechas a imputar
- Rellena los valores ausentes de esas columnas numéricas con la mediana
- Rellena los valores ausentes de esas columnas categóricas con "unknown"
- Y elimina los valores ausentes de esas columnas de tipo fecha

Esto para obtener un DF limpio.

**Llamado desde clean_data**

In [ ]:
# crear función para imputar ausentes

def imputar_segun_diagnostico(df,median_fill_cols,fill_unknown_cols,date_drop_cols):
    # rellenar con la mediana en columnas numéricas
    #El bucle recorre la lista de columnas (median_fill) que queremos rellenar con la mediana.
    #En cada columna, la convertimos a numérica, calculamos su mediana y reemplazamos los valores ausentes con ella.

    for col in median_fill_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        med = df[col].median()
        df[col] = df[col].fillna(med)

    # Agregamos un bucle que recorre las columnas a rellenar los valores ausentes con el texto "unknown"
    # rellenar con texto "unknown" en columnas categóricas
    for col in fill_unknown_cols:
        df[col] = df[col].fillna("unknown")

    # eliminar registros con valores ausentes en columnas tipo fecha
    for col in date_drop_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.dropna(subset=[col]).reset_index(drop=True)
    return df


# Recibir diferentes DataFrames y listas de columnas a procesar:

In [ ]:
# crear función

def clean_data(df,numeric_cols, text_cols, flags_cols,
    median_fill_cols, fill_unknown_cols, date_drop_cols):

    # Sentinels: reemplazar marcadores inválidos por NaN
    df = reemplazar_sentinels_global(df, numeric_cols, text_cols)

    # Flags: crear banderas antes de imputar
    df = crear_flags(df, flags_cols)

    # Imputaciones / drops finales
    df = imputar_segun_diagnostico(df,median_fill_cols,fill_unknown_cols,date_drop_cols)
    return df

In [ ]:
# main_pipeline.py

# DataFrame a procesar: Everpeak
df = pd.read_csv("/datasets/everpeak_retail.csv")

# Listas de columnas - incluir columnas según las necesidades del dataset

# columnas a procesar del dataset EverPeak
## columnas función 1: reemplazar_sentinels
columnas_numericas = ["customer_age"]
columnas_texto = ["product_category"]

## columnas función 2: crear_flags
columnas_flags = ["customer_age", "city", "state"]

## columnas función 3: imputar_segun_diagnostico
cols_imputar_mediana = ["customer_age"]
cols_imputar_unknown = ["city", "state"]
cols_imputar_fecha = ["order_date"]

# aplicar función y guardar resultado en df_clean
df_clean = clean_data(df, columnas_numericas, columnas_texto, columnas_flags,
    cols_imputar_mediana, cols_imputar_unknown, cols_imputar_fecha)

# guardar df_clean en un CSV nuevo
df_clean.to_csv("everpeak_clean.csv", index=False)

In [ ]:
df = pd.read_csv('/datasets/everpeak_clean.csv')

mean_price = df['price'].mean()
median_price = df['price'].median()

print("Promedio:", mean_price)
print("Mediana:", median_price)

In [ ]:
# Graficar histograma
plt.hist(df['price'], bins=50, color='skyblue', edgecolor='black', range=(0,3000))

# Agregar líneas de media y mediana
plt.axvline(mean_price, color='red', linestyle='dashed', linewidth=2, label='Media')
plt.axvline(median_price, color='green', linestyle='dashed', linewidth=2, label='Mediana')

# Títulos y etiquetas
plt.title('Distribución de Precios con Media y Mediana')
plt.xlabel('Precio')
plt.ylabel('Cantidades')
plt.legend(loc='upper left')
plt.show()

# Histograma

In [ ]:
counts, bin_edges, _= plt.hist(df['price'], bins=10, range=(12,1000), color='skyblue', edgecolor='black')
plt.xticks(bin_edges)

plt.title('Histograma de precios con ajuste de eje X')
plt.show()

# Para identificar los outliers de manera más visual, usamos un boxplot:

In [ ]:
sns.boxplot(x=df['order_value'], color='skyblue')
plt.title('Boxplot de order_value')
plt.xlabel('Order value')
plt.show()

In [ ]:
sns.histplot(df['customer_age'], bins=10, color='skyblue', kde=True)
plt.show()

In [ ]:
sns.boxplot(data=df['customer_age'])
plt.show()

# Segmentación para el análisis de clientes

In [ ]:
# Función para clasificar clientes
def classify_segment(row):
    age = row['customer_age']
    spend = row['order_value']

    # Manejo de valores nulos/faltantes
    # pd.isna() verifica de forma robusta si el valor es NaN
    if pd.isna(age) or pd.isna(spend):
        return "Error en Datos"

    # Segmentación de Alto Valor (Gasto >= 10000)
    if spend >= 10000:
        if age >= 55:
            return "Senior VIP"
        else: # age < 55
            return "Junior VIP"

    # Segmentación de Valor Medio (Gasto entre 5000 y 9999)
    elif spend >= 5000:
        if age >= 55:
            return "Sr. Medium Value"
        else: # age < 55
            return "Jr. Medium Value"

    # Segmentación de Valor Bajo (Gasto < 5000)
    else: # spend < 5000
        return "Low Value"

# aplicar función y ver cambios
df["customer_segment"] = df.apply(classify_segment, axis=1)
df[["customer_age", "order_value", "customer_segment"]].head()